# **Data Science and Machine Learning 1 - Year 4 Portfolio**
## Last.Fm API Data Pulling and Cleaning
*By Mark Lambert - C00192497*
*Lecturer: Ben O'Shaughnessey*
*Submission Date: 27th November 2026*

---

In third year I built and deployed a LastFm scrobbling program, which familiarised me with the LastFM API. 

I decided for my Data Science & Machine Learning Portfolio that using this data would be interesting and applicable.
        
This notebook focus' on connecting with the LastFm API using an API key, and then pulling scrobbles using the ```User.getRecentTracks``` and ```User.getLovedTracks``` endpoints.

The data returned requires processing and cleaning. This notebook also covers aggregating the data and merging with an external datasource to expand the features.

```%pip install pandas```

In [3]:
import requests
import pandas as pd

API_KEY = ' api key'
USERNAME = ' username'

all_tracks = []
page = 1
total_pages = 1

while page <= total_pages:
    response = requests.get('http://ws.audioscrobbler.com/2.0/', params={
        'method': 'user.getrecenttracks',
        'user': USERNAME,
        'api_key': API_KEY,
        'format': 'json',
        'limit': 200,
        'page': page
    })
    data = response.json()
    if page == 1:
        total_pages = int(data['recenttracks']['@attr']['totalPages'])
    all_tracks.extend(data['recenttracks']['track'])
    page += 1

The data's raw columns.

The ones that this portfolio will focus on are **artist, album, name and date**. ```@attr``` holds songs that are marked ```nowplaying = true```. 

As such, anything with this attribute will also be ignored later on.

In [12]:
scrobblesResponse = pd.DataFrame(all_tracks)

scrobblesResponse.columns

Index(['artist', 'streamable', 'image', 'mbid', 'album', 'name', '@attr',
       'url', 'date'],
      dtype='str')

The raw data shows that the actual readable details for each artist, album are embedded with a ```#text``` key in a dictionary, which must be extracted

In [13]:
scrobblesResponse

,artist,streamable,image,mbid,album,name,@attr,url,date
0,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",2d2c3253-8b61-3c02-b5e1-83be8ade482c,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Biscuit,{'nowplaying': 'true'},https://www.last.fm/music/Portishead/_/Biscuit,NaN
1,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",7a67d3db-6c7c-3ba9-85d1-357ddf825a7b,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Pedestal,NaN,https://www.last.fm/music/Portishead/_/Pedestal,"{'uts': '1792003542', '#text': '14 Oct 2026, 1..."
2,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",03462159-281b-47ae-9014-81c727ff557e,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Roads,NaN,https://www.last.fm/music/Portishead/_/Roads,"{'uts': '1792003322', '#text': '14 Oct 2026, 1..."
3,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",635649a0-68b4-40d1-a96d-283ed6aa1119,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Numb,NaN,https://www.last.fm/music/Portishead/_/Numb,"{'uts': '1792003018', '#text': '14 Oct 2026, 1..."
4,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",358affb3-cf0f-45ef-bedb-ac4e71fb6403,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Wandering Star,NaN,https://www.last.fm/music/Portishead/_/Wanderi...,"{'uts': '1792002782', '#text': '14 Oct 2026, 1..."
...,...,...,...,...,...,...,...,...,...
18076,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",362c85ba-c79b-478e-8321-0383c3ac8101,{'mbid': '0d2cb66c-3a32-4f81-b977-9231c461c34a...,Why Won't They Talk to Me?,NaN,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,"{'uts': '1763463648', '#text': '18 Nov 2025, 1..."
18077,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",6f93589f-4a47-4a9d-9274-09a0f437ad15,{'mbid': '10ac58ca-0655-4b16-a6cb-58fdc309de0a...,Over,NaN,https://www.last.fm/music/Portishead/_/Over,"{'uts': '1763463370', '#text': '18 Nov 2025, 1..."
18078,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",6da0b598-d9cf-4f62-8376-bc3181e509af,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Obsolete,NaN,https://www.last.fm/music/Tame+Impala/_/Obsolete,"{'uts': '1763462844', '#text': '18 Nov 2025, 1..."
18079,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",ad79b526-8df8-4e79-8756-0607df153e59,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Piece Of Heaven,NaN,https://www.last.fm/music/Tame+Impala/_/Piece+...,"{'uts': '1763462559', '#text': '18 Nov 2025, 1..."


Next, as per [Carlos Medina's article](https://medium.com/hemisferios/using-the-last-fm-api-to-extract-data-and-analyzing-it-with-python-part-1-ffe96f812455), first we must select the colums of interest and, as the column name is obscure I have chosen to rename it to **song**

In [14]:
scrobbles = scrobblesResponse

scrobbles = scrobbles[['artist', 'album', 'name', 'url', 'date']]
scrobbles.rename(columns={'name': 'song'}, inplace=True)
scrobbles

,artist,album,song,url,date
0,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Biscuit,https://www.last.fm/music/Portishead/_/Biscuit,NaN
1,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,"{'uts': '1792003542', '#text': '14 Oct 2026, 1..."
2,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Roads,https://www.last.fm/music/Portishead/_/Roads,"{'uts': '1792003322', '#text': '14 Oct 2026, 1..."
3,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Numb,https://www.last.fm/music/Portishead/_/Numb,"{'uts': '1792003018', '#text': '14 Oct 2026, 1..."
4,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '00163fb1-5ab7-4fec-b0dd-a915e2722340...,Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,"{'uts': '1792002782', '#text': '14 Oct 2026, 1..."
...,...,...,...,...,...
18076,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '0d2cb66c-3a32-4f81-b977-9231c461c34a...,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,"{'uts': '1763463648', '#text': '18 Nov 2025, 1..."
18077,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '10ac58ca-0655-4b16-a6cb-58fdc309de0a...,Over,https://www.last.fm/music/Portishead/_/Over,"{'uts': '1763463370', '#text': '18 Nov 2025, 1..."
18078,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,"{'uts': '1763462844', '#text': '18 Nov 2025, 1..."
18079,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,"{'uts': '1763462559', '#text': '18 Nov 2025, 1..."


Next, convert the ```artist``` ```album``` and ```date``` to strings

In [15]:
scrobbles['artist'] = scrobbles['artist'].astype(str)
scrobbles['album'] = scrobbles['album'].astype(str)
scrobbles['date'] = scrobbles['date'].astype(str)

Extract the embedded ```#text``` attribute from each of the ```artist, album``` and ```date``` fields

In [16]:
scrobbles[['mbid2','artist']]=scrobbles['artist'].str.split("#text':", n = 1, expand = True)
scrobbles= scrobbles.drop('mbid2', axis=1)
scrobbles[['mbid3','album']]=scrobbles['album'].str.split("#text':", n = 1, expand = True)
scrobbles= scrobbles.drop('mbid3', axis=1)
scrobbles[['uts','date']]=scrobbles['date'].str.split("#text':", n = 1, expand = True)
scrobbles.head()

,artist,album,song,url,date,uts
0,'Portishead'},'Dummy'},Biscuit,https://www.last.fm/music/Portishead/_/Biscuit,NaN,NaN
1,'Portishead'},'Dummy'},Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,"'14 Oct 2026, 18:45'}","{'uts': '1792003542', '"
2,'Portishead'},'Dummy'},Roads,https://www.last.fm/music/Portishead/_/Roads,"'14 Oct 2026, 18:42'}","{'uts': '1792003322', '"
3,'Portishead'},'Dummy'},Numb,https://www.last.fm/music/Portishead/_/Numb,"'14 Oct 2026, 18:36'}","{'uts': '1792003018', '"
4,'Portishead'},'Dummy'},Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,"'14 Oct 2026, 18:33'}","{'uts': '1792002782', '"


Removing the ```}``` and ```'``` characters from the ```artist and album``` entries

In [17]:
scrobbles['artist'] = scrobbles['artist'].str.strip("}'\" ")
scrobbles['album'] = scrobbles['album'].str.strip("}'\" ")
scrobbles.head()

,artist,album,song,url,date,uts
0,Portishead,Dummy,Biscuit,https://www.last.fm/music/Portishead/_/Biscuit,NaN,NaN
1,Portishead,Dummy,Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,"'14 Oct 2026, 18:45'}","{'uts': '1792003542', '"
2,Portishead,Dummy,Roads,https://www.last.fm/music/Portishead/_/Roads,"'14 Oct 2026, 18:42'}","{'uts': '1792003322', '"
3,Portishead,Dummy,Numb,https://www.last.fm/music/Portishead/_/Numb,"'14 Oct 2026, 18:36'}","{'uts': '1792003018', '"
4,Portishead,Dummy,Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,"'14 Oct 2026, 18:33'}","{'uts': '1792002782', '"


The ```date``` and ```uts``` fields also need to be stripped of the remaining dirty characters. I have also chosen to rename uts to **timestamp** as it is more descriptive.

In [18]:
scrobbles['date'] = scrobbles['date'].str.strip("}'\" ")
scrobbles['uts'] = scrobbles['uts'].str.strip("{uts':','\" ")
scrobbles.rename(columns={'uts': 'timestamp'}, inplace=True)
scrobbles.head()

,artist,album,song,url,date,timestamp
0,Portishead,Dummy,Biscuit,https://www.last.fm/music/Portishead/_/Biscuit,NaN,NaN
1,Portishead,Dummy,Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,"14 Oct 2026, 18:45",1792003542
2,Portishead,Dummy,Roads,https://www.last.fm/music/Portishead/_/Roads,"14 Oct 2026, 18:42",1792003322
3,Portishead,Dummy,Numb,https://www.last.fm/music/Portishead/_/Numb,"14 Oct 2026, 18:36",1792003018
4,Portishead,Dummy,Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,"14 Oct 2026, 18:33",1792002782


Next, the date field must be converted into a date only field

In [19]:
scrobbles['date'] = pd.to_datetime(scrobbles['date'], unit='s')
scrobbles['date_only'] = scrobbles['date'].dt.date
scrobbles

,artist,album,song,url,date,timestamp,date_only
0,Portishead,Dummy,Biscuit,https://www.last.fm/music/Portishead/_/Biscuit,NaT,NaN,NaT
1,Portishead,Dummy,Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,2026-10-14 18:45:00,1792003542,2026-10-14
2,Portishead,Dummy,Roads,https://www.last.fm/music/Portishead/_/Roads,2026-10-14 18:42:00,1792003322,2026-10-14
3,Portishead,Dummy,Numb,https://www.last.fm/music/Portishead/_/Numb,2026-10-14 18:36:00,1792003018,2026-10-14
4,Portishead,Dummy,Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,2026-10-14 18:33:00,1792002782,2026-10-14
...,...,...,...,...,...,...,...
18076,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18
18077,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18
18078,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18
18079,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18


Finally, the row which contains NaN must be removed, as this was a song that had the ```@attr``` of ```nowplaying = true```, so it should not count as a fully scrobbled song

In [20]:
scrobbles = scrobbles.dropna(subset=['date'])
scrobbles

,artist,album,song,url,date,timestamp,date_only
1,Portishead,Dummy,Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,2026-10-14 18:45:00,1792003542,2026-10-14
2,Portishead,Dummy,Roads,https://www.last.fm/music/Portishead/_/Roads,2026-10-14 18:42:00,1792003322,2026-10-14
3,Portishead,Dummy,Numb,https://www.last.fm/music/Portishead/_/Numb,2026-10-14 18:36:00,1792003018,2026-10-14
4,Portishead,Dummy,Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,2026-10-14 18:33:00,1792002782,2026-10-14
5,Portishead,Dummy,It Could Be Sweet,https://www.last.fm/music/Portishead/_/It+Coul...,2026-10-14 18:28:00,1792002489,2026-10-14
...,...,...,...,...,...,...,...
18076,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18
18077,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18
18078,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18
18079,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18


Then, three new columns- ```hour``` ```tod``` ```month``` are added. First, the time has to be converted to Dublin's time. Then, we can call the time_of_day function to classify between the four periods of a day: **morning, afternoon, evening and night.**

In [21]:
def time_of_day(hour):
    if hour >= 22 or hour < 5:
        return 'night'
    if hour < 12:
        return 'morning'
    if hour < 17:
        return 'afternoon'
    return 'evening'

local_time = scrobbles['date'].dt.tz_localize('UTC').dt.tz_convert('Europe/Dublin')
scrobbles['hour'] = local_time.dt.hour
scrobbles['tod'] = scrobbles['hour'].apply(time_of_day)
scrobbles['month'] = scrobbles['date'].dt.to_period('M')
scrobbles.head()

,artist,album,song,url,date,timestamp,date_only,hour,tod,month
1,Portishead,Dummy,Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,2026-10-14 18:45:00,1792003542,2026-10-14,19,evening,2026-10
2,Portishead,Dummy,Roads,https://www.last.fm/music/Portishead/_/Roads,2026-10-14 18:42:00,1792003322,2026-10-14,19,evening,2026-10
3,Portishead,Dummy,Numb,https://www.last.fm/music/Portishead/_/Numb,2026-10-14 18:36:00,1792003018,2026-10-14,19,evening,2026-10
4,Portishead,Dummy,Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,2026-10-14 18:33:00,1792002782,2026-10-14,19,evening,2026-10
5,Portishead,Dummy,It Could Be Sweet,https://www.last.fm/music/Portishead/_/It+Coul...,2026-10-14 18:28:00,1792002489,2026-10-14,19,evening,2026-10


To get my top 10 artists, we can query the dataset by taking a total count per artist and slect the top 10

Aggregate the data into daily scrobbles

In [22]:
daily = scrobbles.groupby('date_only').size().reset_index(name='scrobble_count')
daily['date_only'] = pd.to_datetime(daily['date_only'])
daily.set_index('date_only', inplace=True)

In [23]:
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
for i in range(7):
    daily[days[i]] = (daily.index.dayofweek == i).astype(float)

In [24]:
monthly = scrobbles.groupby('month').size().reset_index(name='scrobble_count')
monthly

,month,scrobble_count
0,2025-11,1254
1,2025-12,1732
2,2026-01,1692
3,2026-02,1345
4,2026-03,1022
5,2026-04,522
6,2026-05,829
7,2026-06,1942
8,2026-07,2479
9,2026-08,2430


Output the data to csv's so that I don't have to request the data each time I run the program

In [25]:
scrobbles.to_csv('datasets/scrobbles.csv', index=False)

In [26]:
daily.to_csv('datasets/daily.csv', index=True)

Using the [unofficial docs](https://lastfm-docs.github.io/api-docs/user/getLovedTracks/#examples) once again, the data can be combined with the ```User.getLovedTracks``` endpoint. A new field called ```is_loved``` can be joined to the full scrobble dataset

In [27]:
all_loved_tracks = []
loved_page = 1
total_loved_pages = 1

while loved_page <= total_loved_pages:
    response = requests.get('http://ws.audioscrobbler.com/2.0/', params={
        'method': 'user.getlovedtracks',
        'user': USERNAME,
        'api_key': API_KEY,
        'format': 'json',
        'limit': 200,
        'page': loved_page
    })
    data = response.json()
    if loved_page == 1:
        total_loved_pages = int(data['lovedtracks']['@attr']['totalPages'])
    tracks = data['lovedtracks']['track']
    for track in tracks:
        all_loved_tracks.append({
            'artist': track['artist']['name'],
            'track': track['name'],
            'timestamp': int(track['date']['uts']) if 'date' in track else None
        })
    loved_page += 1

Merging the scrobbles and retrieved lovedScrobbles tracks on the columns ```artist``` and ```song``` against ```track``` and adding the new col ```is_loved``` 

In [28]:
lovedScrobbles = pd.DataFrame(all_loved_tracks)
lovedScrobbles['is_loved'] = 1
lovedScrobbles = lovedScrobbles.drop_duplicates(subset=['artist', 'track'])

lovedMerged = pd.merge(
    scrobbles,
    lovedScrobbles[['artist', 'track', 'is_loved']],
    left_on=['artist', 'song'],
    right_on=['artist', 'track'],
    how='left'
)

loved = lovedMerged.drop(columns=['track'])
loved['is_loved'] = loved['is_loved'].fillna(0).astype(int)
loved

,artist,album,song,url,date,timestamp,date_only,hour,tod,month,is_loved
0,Portishead,Dummy,Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,2026-10-14 18:45:00,1792003542,2026-10-14,19,evening,2026-10,0
1,Portishead,Dummy,Roads,https://www.last.fm/music/Portishead/_/Roads,2026-10-14 18:42:00,1792003322,2026-10-14,19,evening,2026-10,1
2,Portishead,Dummy,Numb,https://www.last.fm/music/Portishead/_/Numb,2026-10-14 18:36:00,1792003018,2026-10-14,19,evening,2026-10,0
3,Portishead,Dummy,Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,2026-10-14 18:33:00,1792002782,2026-10-14,19,evening,2026-10,1
4,Portishead,Dummy,It Could Be Sweet,https://www.last.fm/music/Portishead/_/It+Coul...,2026-10-14 18:28:00,1792002489,2026-10-14,19,evening,2026-10,0
...,...,...,...,...,...,...,...,...,...,...,...
17986,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18,11,morning,2025-11,1
17987,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18,10,morning,2025-11,1
17988,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18,10,morning,2025-11,1
17989,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18,10,morning,2025-11,0


In [29]:
loved.to_csv('datasets/loved.csv', index=False)

In [30]:
spotify = pd.read_csv('datasets/spotify.csv')

Using a Spotify track dataset from [Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset) which contains additional features such as ```danceability```, ```energy``` and ```tempo```

In order to get a higher match rate, the artist and track features are stripped and lowercase 

In [31]:
spotify['clean_artist'] = spotify['artists'].str.lower().str.strip()
spotify['clean_track'] = spotify['track_name'].str.lower().str.strip()

loved['clean_artist'] = loved['artist'].str.lower().str.strip()
loved['clean_track'] = loved['song'].str.lower().str.strip()

spotify_unique = spotify.drop_duplicates(subset=['clean_artist', 'clean_track'])

spotifyMerged = pd.merge(
    loved,
    spotify_unique,
    on=['clean_artist', 'clean_track'],
    how='left',
    indicator=True
)

matched_count = (spotifyMerged['_merge'] == 'both').sum()
total_count = len(loved)
match_rate = (matched_count / total_count) * 100

print(f"{matched_count} of {total_count} tracks match ({match_rate:.2f}%)")

spotifyMerged

6308 of 17991 tracks match (35.06%)


,artist,album,song,url,date,timestamp,date_only,hour,tod,month,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,_merge
0,Portishead,Dummy,Pedestal,https://www.last.fm/music/Portishead/_/Pedestal,2026-10-14 18:45:00,1792003542,2026-10-14,19,evening,2026-10,...,0.0,0.0760,0.154,0.721000,0.1170,0.377,84.125,4.0,trip-hop,both
1,Portishead,Dummy,Roads,https://www.last.fm/music/Portishead/_/Roads,2026-10-14 18:42:00,1792003322,2026-10-14,19,evening,2026-10,...,0.0,0.0761,0.778,0.168000,0.1060,0.193,76.110,4.0,electronic,both
2,Portishead,Dummy,Numb,https://www.last.fm/music/Portishead/_/Numb,2026-10-14 18:36:00,1792003018,2026-10-14,19,evening,2026-10,...,1.0,0.0356,0.056,0.000018,0.0812,0.220,78.052,4.0,trip-hop,both
3,Portishead,Dummy,Wandering Star,https://www.last.fm/music/Portishead/_/Wanderi...,2026-10-14 18:33:00,1792002782,2026-10-14,19,evening,2026-10,...,1.0,0.0723,0.504,0.587000,0.1170,0.707,80.152,4.0,trip-hop,both
4,Portishead,Dummy,It Could Be Sweet,https://www.last.fm/music/Portishead/_/It+Coul...,2026-10-14 18:28:00,1792002489,2026-10-14,19,evening,2026-10,...,1.0,0.0444,0.698,0.016500,0.1300,0.556,153.963,4.0,trip-hop,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17986,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18,11,morning,2025-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
17987,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18,10,morning,2025-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
17988,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18,10,morning,2025-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
17989,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18,10,morning,2025-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


We make a new dataset called ```aggregated```

First we take one row per unique track w/ the spotify features selected in ```audio_cols```. 
Then we add the ```play_count``` for each track and remove any tracks that didn't match between the two datasets

The result is one row per matched track with the audio features and play count for each track. 

In [32]:
audio_cols = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'tempo', 'speechiness']

aggregated = spotifyMerged.groupby(['clean_artist', 'clean_track'])[audio_cols].first().reset_index()
aggregated['play_count'] = spotifyMerged.groupby(['clean_artist', 'clean_track'])['is_loved'].size().values
aggregated = aggregated.dropna(subset=['danceability', 'energy', 'valence'])
aggregated.to_csv('datasets/aggregatedData.csv', index=False)
aggregated

,clean_artist,clean_track,danceability,energy,valence,acousticness,instrumentalness,tempo,speechiness,play_count
4,a família,brinquedo assassino,0.771,0.746,0.502,0.090200,0.000000,148.054,0.2620,2
13,abba,super trouper,0.724,0.724,0.927,0.294000,0.000000,117.986,0.0304,1
14,adnan sami,tera chehra,0.646,0.562,0.488,0.523000,0.002100,100.000,0.0331,1
20,air,kelly watch the stars,0.516,0.743,0.226,0.282000,0.637000,109.837,0.0292,17
21,air,la femme d'argent,0.517,0.737,0.558,0.434000,0.684000,79.820,0.0352,20
...,...,...,...,...,...,...,...,...,...,...
2871,weezer,pork and beans,0.599,0.626,0.708,0.010800,0.000009,119.837,0.0339,1
2872,weezer,say it ain't so,0.635,0.551,0.451,0.007430,0.000018,151.779,0.0354,2
2874,weezer,the world has turned and left me here,0.455,0.918,0.560,0.000399,0.013400,99.078,0.0352,1
2875,weezer,undone - the sweater song,0.457,0.618,0.579,0.001140,0.002570,80.184,0.0296,1
